<!-- Curated copy -->
> **Curated copy.** This notebook is taken verbatim from the BTech-thesis working archive; only
> cell *outputs* have been cleared and machine-specific absolute paths (`C:\\...`, `D:\\...`,
> `F:\\...`) have been rewritten to repository-relative `runs/...` paths. No scientific logic,
> equation, hyper-parameter or architecture has been modified. Place regenerated
> `dataset_run_*` folders under a `runs/` directory next to this notebook (or edit the paths).
> The figures this notebook originally produced are preserved in the sibling `figures/` folder.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import os, json,ast, math, glob, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import glob
import pandas as pd
from sklearn.decomposition import PCA
import importlib

In [ ]:
# ============================================================
# 1) PHYSICS / DOMAIN SETTINGS
# ============================================================
N = 40                         # 3D grid size along each axis (Nx = Ny = Nz = N)
Lx = Ly = Lz = 0.05            # domain lengths [m]

rho = 800.0                    # density [kg/m^3]
cp = 2000.0                    # specific heat [J/(kg.K)]
k = 0.2                        # thermal conductivity [W/(m.K)]
L_lat = 2e5                    # latent heat [J/kg]

T_m = 330.0                    # melting temperature [K]
T_init = 300.0                 # initial temperature [K]
T_bound = 330.0                # reference boundary temperature [K]

t_end = 4000.0                 # total simulation time [s]
save_times = (
    100.0, 250.0, 400.0, 600.0,
    1000.0, 1200.0, 1500.0, 1800.0, 2100.0
)                              # time snapshots saved from CFD
cfl = 0.30                    # CFL number for numerical stability


# ============================================================
# 2) VOLUMETRIC HEAT-SOURCE GENERATION
# ============================================================
q_scale = 1e5                  # heat-source magnitude scale [W/m^3]
Q_length_scale = 0.18          # GP/RBF length scale for source smoothness
Q_sigma = 1.0                  # GP/RBF amplitude scale


# ============================================================
# 3) BOUNDARY-CONDITION RANDOMIZATION
# ============================================================
mu_max = 0.8                   # max center-location parameter for BC profile
sigma_max = 0.7                # max spread parameter for BC profile
amp_max = 80                   # max BC amplitude variation

only_lr_vary = True            # if True, vary only left/right boundaries
All_side_const_temp_boundary = False
# if True, use constant temperature on all sides for testing/debugging


# ============================================================
# 4) DATASET SETTINGS
# ============================================================
NUM_CASES = 500                # total number of CFD/generated cases
TRAIN_FRAC = 0.8               # training split fraction
VAL_FRAC = 0.1                 # validation split fraction
# test fraction = 1 - TRAIN_FRAC - VAL_FRAC

TARGET = "T"                   # "T" for temperature, "f" for liquid fraction


# ============================================================
# 5) TRAINING SETTINGS
# ============================================================
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0               # can increase later for faster loading
VAL_SAMPLES = 4                # number of validation cases/slices to visualize/check
WINDOW_SIZE = 13
WINDOW_RADIUS = WINDOW_SIZE // 2 
TIME_DIM = 64
TIME_N_FREQ = 8

# ------------------------------------------------------------
# LOSS WEIGHTS
# ------------------------------------------------------------
SMOOTHNESS_WEIGHT = 1e-4
GRAD_WEIGHT       = 0.15
INTERFACE_WEIGHT  = 0.2 if TARGET=="f" else 0  # used only if TARGET == "f"

INTERFACE_CENTER = 0.5
INTERFACE_SIGMA  = 0.15
INTERFACE_ALPHA  = 2.0

# ============================================================
# 6) UNIFIED 2D SLICE U-NET SETTINGS
# ============================================================
IN_CHANNELS = 17               # fixed number of input channels per 2D slice
UNET_FEATURES = 64             # base feature width
UNET_DROPOUT = 0.0
ACTIVATION_FUNCTION = "sin"    # keep only if your custom blocks actually use this


In [ ]:
# ============================================================
# 0) DATASET RUN DIRECTORY + DEVICE SETUP
# ============================================================

ROOT = Path(".")   # parent folder containing dataset_run_* directories

def latest_run_dir(root: Path) -> Path | None:
    """Return the most recent dataset_run_* directory inside root."""
    run_dirs = sorted(
        [Path(p) for p in glob.glob(str(root / "dataset_run_*")) if Path(p).is_dir()]
    )
    return run_dirs[-1] if run_dirs else None


RUN_DIR = latest_run_dir(ROOT)
assert RUN_DIR is not None, "No dataset_run_* folder found. Generate the dataset first."

print(f"Using RUN_DIR: {RUN_DIR}")

# Main files expected inside RUN_DIR
DATASET_FILE = RUN_DIR / "slice_dataset_3d.npz"   # update this if your saved filename differs
SPLITS_FILE  = RUN_DIR / "splits.json"

assert DATASET_FILE.exists(), f"Dataset file not found: {DATASET_FILE}"
assert SPLITS_FILE.exists(),  f"Splits file not found: {SPLITS_FILE}"

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")

In [ ]:
# ============================================================
# VERIFY DATASET FILES
# ============================================================

npz_path = DATASET_FILE
splits_path = SPLITS_FILE

assert npz_path.exists(), f"Missing dataset file: {npz_path}"
assert splits_path.exists(), f"Missing splits file: {splits_path}"

print("Found dataset:", npz_path.resolve())
print("Found splits :", splits_path.resolve())

In [ ]:
# ============================================================
# LOAD UPDATED 3D SEQUENCE SLICE DATASET
# ============================================================

npz = np.load(npz_path, allow_pickle=True)

# Axis-wise STATIC slice inputs and SEQUENCE targets
Xstatic_x = npz["Xstatic_x"].astype(np.float32)   # [C, Sx, Cin, H, W]
Y_x       = npz["Y_x"].astype(np.float32)         # [C, Sx, T, 2, H, W]

Xstatic_y = npz["Xstatic_y"].astype(np.float32)   # [C, Sy, Cin, H, W]
Y_y       = npz["Y_y"].astype(np.float32)         # [C, Sy, T, 2, H, W]

Xstatic_z = npz["Xstatic_z"].astype(np.float32)   # [C, Sz, Cin, H, W]
Y_z       = npz["Y_z"].astype(np.float32)         # [C, Sz, T, 2, H, W]

# Optional full 3D fields
Q_all = npz["Q_all"].astype(np.float32) if "Q_all" in npz.files else None
T_all = npz["T_all"].astype(np.float32) if "T_all" in npz.files else None
f_all = npz["f_all"].astype(np.float32) if "f_all" in npz.files else None

# Time values and bookkeeping
times_np    = npz["times"].astype(np.float32) if "times" in npz.files else np.array(save_times, dtype=np.float32)
case_ids_np = npz["case_ids"].astype(np.int64) if "case_ids" in npz.files else np.arange(Xstatic_x.shape[0], dtype=np.int64)
meta        = npz["meta"].item() if "meta" in npz.files else {}

# ------------------------------------------------------------
# Basic dimensions
# ------------------------------------------------------------
C_x, Sx, Cin_x, Hx, Wx = Xstatic_x.shape
C_y, Sy, Cin_y, Hy, Wy = Xstatic_y.shape
C_z, Sz, Cin_z, Hz, Wz = Xstatic_z.shape

C_yx, Sx_y, T,   Cout_x, Hx_y, Wx_y = Y_x.shape
C_yy, Sy_y, T_y, Cout_y, Hy_y, Wy_y = Y_y.shape
C_zy, Sz_y, T_z, Cout_z, Hz_y, Wz_y = Y_z.shape

assert C_x == C_y == C_z, "Mismatch in number of cases across Xstatic_x, Xstatic_y, Xstatic_z"
assert C_yx == C_x and C_yy == C_y and C_zy == C_z, "Mismatch between Xstatic_* and Y_* case counts"

assert Sx == Sx_y, "Mismatch between Xstatic_x and Y_x slice counts"
assert Sy == Sy_y, "Mismatch between Xstatic_y and Y_y slice counts"
assert Sz == Sz_y, "Mismatch between Xstatic_z and Y_z slice counts"

assert Hx == Hx_y and Wx == Wx_y, "Mismatch between Xstatic_x and Y_x spatial dimensions"
assert Hy == Hy_y and Wy == Wy_y, "Mismatch between Xstatic_y and Y_y spatial dimensions"
assert Hz == Hz_y and Wz == Wz_y, "Mismatch between Xstatic_z and Y_z spatial dimensions"

assert T == T_y == T_z, "Mismatch in number of time snapshots across axes"
assert Cin_x == Cin_y == Cin_z, "Mismatch in input channel count across axes"
assert Cout_x == Cout_y == Cout_z == 2, "Target channel dimension must be 2: [T, f]"
assert len(times_np) == T, "times_np length must match target sequence length T"

Cin = Cin_x
Cout = Cout_x
NUM_CASES = C_x

# ------------------------------------------------------------
# Window configuration (for multi-slice learning)
# ------------------------------------------------------------

assert WINDOW_SIZE % 2 == 1, "WINDOW_SIZE must be odd"

print("\nWindow config:")
print("WINDOW_SIZE  :", WINDOW_SIZE)
print("WINDOW_RADIUS:", WINDOW_RADIUS)

# ------------------------------------------------------------
# (Optional) Helper for debugging slice windows
# ------------------------------------------------------------
def debug_window_indices(center_idx, n_slices):
    ids = []
    for off in range(-WINDOW_RADIUS, WINDOW_RADIUS + 1):
        j = center_idx + off
        j = max(0, min(n_slices - 1, j))  # clamp
        ids.append(j)
    return ids

# Example debug (first few slices)
print("\nSample window indices (x-axis):")
for i in range(min(3, Sx)):
    print(f"center {i} ->", debug_window_indices(i, Sx))

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------
print("\nData summary:")
print("Xstatic_x :", Xstatic_x.shape)
print("Y_x       :", Y_x.shape)
print("Xstatic_y :", Xstatic_y.shape)
print("Y_y       :", Y_y.shape)
print("Xstatic_z :", Xstatic_z.shape)
print("Y_z       :", Y_z.shape)

if Q_all is not None:
    print("Q_all     :", Q_all.shape)
if T_all is not None:
    print("T_all     :", T_all.shape)
if f_all is not None:
    print("f_all     :", f_all.shape)

print("times     :", times_np.shape, times_np[:min(5, len(times_np))], "...")
print("case_ids  :", case_ids_np.shape)
print("Cin       :", Cin)
print("Cout      :", Cout)
print("NUM_CASES :", NUM_CASES)
print("meta keys :", list(meta.keys()))

In [ ]:
# ============================================================
# LOAD TRAIN / VAL / TEST SPLITS
# ============================================================

with open(splits_path, "r") as f:
    splits = json.load(f)

train_ids = np.array(splits["train"], dtype=np.int64)
val_ids   = np.array(splits["val"], dtype=np.int64)
test_ids  = np.array(splits["test"], dtype=np.int64)

print("Split sizes:", len(train_ids), len(val_ids), len(test_ids))


In [ ]:
def compute_train_stats_unified(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    train_ids,
    target="f",
    eps=1e-6
):
    """
    Compute normalization stats for SEQUENCE dataset.

    Xstatic_* : [C, S, Cin, H, W]
    Y_*       : [C, S, T, 2, H, W]
    """

    tgt_idx = 0 if target == "T" else 1

    # --------------------------------------------------------
    # Select training cases
    # --------------------------------------------------------
    Xx_tr = Xstatic_x[train_ids]   # [Ctr, Sx, Cin, H, W]
    Xy_tr = Xstatic_y[train_ids]   # [Ctr, Sy, Cin, H, W]
    Xz_tr = Xstatic_z[train_ids]   # [Ctr, Sz, Cin, H, W]

    Yx_tr = Y_x[train_ids, :, :, tgt_idx]   # [Ctr, Sx, T, H, W]
    Yy_tr = Y_y[train_ids, :, :, tgt_idx]   # [Ctr, Sy, T, H, W]
    Yz_tr = Y_z[train_ids, :, :, tgt_idx]   # [Ctr, Sz, T, H, W]

    # --------------------------------------------------------
    # INPUT STATS
    # per-channel mean/std across:
    # (case, slice, height, width)
    # --------------------------------------------------------
    x_mean_x = Xx_tr.mean(axis=(0, 1, 3, 4))   # [Cin]
    x_mean_y = Xy_tr.mean(axis=(0, 1, 3, 4))
    x_mean_z = Xz_tr.mean(axis=(0, 1, 3, 4))

    x_std_x = Xx_tr.std(axis=(0, 1, 3, 4))
    x_std_y = Xy_tr.std(axis=(0, 1, 3, 4))
    x_std_z = Xz_tr.std(axis=(0, 1, 3, 4))

    # average across axes
    x_mean = (x_mean_x + x_mean_y + x_mean_z) / 3.0
    x_std  = (x_std_x  + x_std_y  + x_std_z)  / 3.0
    x_std  = x_std + eps

    # reshape for broadcasting
    x_mean = x_mean[None, :, None, None].astype(np.float32)
    x_std  = x_std[None, :, None, None].astype(np.float32)

    # --------------------------------------------------------
    # OUTPUT STATS
    # across:
    # (case, slice, time, height, width)
    # --------------------------------------------------------
    y_sum = (
        Yx_tr.astype(np.float64).sum() +
        Yy_tr.astype(np.float64).sum() +
        Yz_tr.astype(np.float64).sum()
    )

    y_count = (
        Yx_tr.size +
        Yy_tr.size +
        Yz_tr.size
    )

    y_mean = y_sum / y_count

    y_sq_sum = (
        ((Yx_tr.astype(np.float64) - y_mean) ** 2).sum() +
        ((Yy_tr.astype(np.float64) - y_mean) ** 2).sum() +
        ((Yz_tr.astype(np.float64) - y_mean) ** 2).sum()
    )

    y_std = np.sqrt(y_sq_sum / y_count) + eps

    return {
        "x_mean": x_mean,
        "x_std": x_std,
        "y_mean": float(y_mean),
        "y_std": float(y_std),
    }

# ------------------------------------------------------------
# Compute stats for current target mode
# TARGET = "T" or "f"
# ------------------------------------------------------------
stats = compute_train_stats_unified(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    train_ids=train_ids,
    target=TARGET
)

print("Target mode :", TARGET)
print("Y mean/std  :", stats["y_mean"], stats["y_std"])
print("X mean/std shapes:", stats["x_mean"].shape, stats["x_std"].shape)

In [ ]:
class Unified3DSliceDataset(Dataset):
    """
    Unified dataset for 3D slice SEQUENCE learning with one common 2D model.

    One sample corresponds to:
        (case_id, axis, slice_id)

    Expected input arrays
    ---------------------
    Xstatic_x, Xstatic_y, Xstatic_z : np.ndarray
        Shape: [C, S, Cin, H, W]

    Y_x, Y_y, Y_z : np.ndarray
        Shape: [C, S, T, 2, H, W]
        target channel 0 -> temperature
        target channel 1 -> liquid fraction

    times : np.ndarray
        Shape: [T]

    ids : array-like
        Case indices to include in this dataset split

    stats : dict
        Must contain:
            "x_mean": [1, Cin, 1, 1]
            "x_std" : [1, Cin, 1, 1]
            "y_mean": scalar
            "y_std" : scalar

    target : str
        "T" for temperature
        "f" for liquid fraction
    """

    AXIS_TO_ID = {"x": 0, "y": 1, "z": 2}
    ID_TO_AXIS = {0: "x", 1: "y", 2: "z"}

    def __init__(
        self,
        Xstatic_x, Y_x,
        Xstatic_y, Y_y,
        Xstatic_z, Y_z,
        times,
        ids,
        stats,
        target="T",
        window_size=3,
        window_mode="clamp"
    ):
        super().__init__()

        assert target in ["T", "f"]
        assert window_size % 2 == 1, "window_size must be odd"

        self.X_by_axis = {
            "x": Xstatic_x,
            "y": Xstatic_y,
            "z": Xstatic_z,
        }
        self.Y_by_axis = {
            "x": Y_x,
            "y": Y_y,
            "z": Y_z,
        }

        self.times = np.asarray(times, dtype=np.float32)
        self.ids = np.asarray(ids, dtype=np.int64)
        self.stats = stats
        self.target = target
        self.target_idx = 0 if target == "T" else 1

        self.t_scale = float(np.max(self.times)) if float(np.max(self.times)) > 0.0 else 1.0

        self.x_mean = self.stats["x_mean"][0].astype(np.float32)
        self.x_std  = self.stats["x_std"][0].astype(np.float32)
        self.y_mean = float(self.stats["y_mean"])
        self.y_std  = float(self.stats["y_std"])

        self.window_size = window_size
        self.window_radius = window_size // 2
        self.window_mode = window_mode

        # ---- Build samples ----
        self.samples = []

        for case_id in self.ids:
            for axis_name in ["x", "y", "z"]:
                X_axis = self.X_by_axis[axis_name]
                _, S_axis, _, _, _ = X_axis.shape

                for slice_idx in range(S_axis):
                    self.samples.append({
                        "case_id": int(case_id),
                        "axis_name": axis_name,
                        "axis_id": self.AXIS_TO_ID[axis_name],
                        "slice_idx": int(slice_idx),
                    })

    def __len__(self):
        return len(self.samples)

    # --------------------------------------------------------
    # NEW: window index helper
    # --------------------------------------------------------
    def _get_window_ids(self, center_idx, n_slices):
        ids = []

        for off in range(-self.window_radius, self.window_radius + 1):
            j = center_idx + off

            if self.window_mode == "clamp":
                j = max(0, min(n_slices - 1, j))

            elif self.window_mode == "reflect":
                if j < 0:
                    j = -j
                if j >= n_slices:
                    j = 2 * n_slices - 2 - j
                j = max(0, min(n_slices - 1, j))

            else:
                raise ValueError("Unknown window_mode")

            ids.append(j)

        return ids

    # --------------------------------------------------------
    # MAIN CHANGE HERE
    # --------------------------------------------------------
    def __getitem__(self, i):

        info = self.samples[i]

        case_id   = info["case_id"]
        axis_name = info["axis_name"]
        axis_id   = info["axis_id"]
        slice_idx = info["slice_idx"]

        X_axis = self.X_by_axis[axis_name]   # [C,S,Cin,H,W]
        Y_axis = self.Y_by_axis[axis_name]   # [C,S,T,2,H,W]

        _, S_axis, Cin, H, W = X_axis.shape

        # --------------------------------------------------------
        # NEW: MULTI-SLICE WINDOW
        # --------------------------------------------------------
        win_ids = self._get_window_ids(slice_idx, S_axis)

        X_list = []
        for j in win_ids:
            Xj = X_axis[case_id, j].astype(np.float32)
            Xj = (Xj - self.x_mean) / self.x_std
            X_list.append(Xj)

        X_window = np.stack(X_list, axis=0).astype(np.float32)   # [Wn,Cin,H,W]

        # --------------------------------------------------------
        # TARGET (center slice only)
        # --------------------------------------------------------
        Y = Y_axis[case_id, slice_idx, :, self.target_idx].astype(np.float32)
        Y_norm = (Y - self.y_mean) / self.y_std

        # --------------------------------------------------------
        # TIME
        # --------------------------------------------------------
        times_raw = self.times.astype(np.float32)
        times_norm = (times_raw / self.t_scale).astype(np.float32)

        return {
            "X_window": torch.from_numpy(X_window),    # [Wn,Cin,H,W]
            "times": torch.from_numpy(times_norm),     # [T]
            "times_raw": torch.from_numpy(times_raw),

            "Y": torch.from_numpy(Y_norm),             # [T,H,W]
            "Y_raw": torch.from_numpy(Y),

            "case_index": int(case_id),
            "axis_id": int(axis_id),
            "axis_name": axis_name,
            "slice_index": int(slice_idx),
            "window_ids": torch.tensor(win_ids, dtype=torch.long),
        }

In [ ]:
# ============================================================
# BUILD DATASETS
# ============================================================
ds_train = Unified3DSliceDataset(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    times=times_np,
    ids=train_ids,
    stats=stats,
    target=TARGET,
    window_size=WINDOW_SIZE
)

ds_val = Unified3DSliceDataset(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    times=times_np,
    ids=val_ids,
    stats=stats,
    target=TARGET,
    window_size=WINDOW_SIZE
)

# ============================================================
# BUILD DATALOADERS
# ============================================================

dl_train = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

dl_val = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ============================================================
# SANITY CHECK
# ============================================================

batch = next(iter(dl_train))

print("Batch keys   :", batch.keys())

# NEW INPUT
print("X_window     :", batch["X_window"].shape)   # [B, Wn, Cin, H, W]

print("times        :", batch["times"].shape)      # [B, T]
print("times_raw    :", batch["times_raw"].shape)  # [B, T]

print("Y            :", batch["Y"].shape)          # [B, T, H, W]
print("Y_raw        :", batch["Y_raw"].shape)      # [B, T, H, W]

print("case_index   :", batch["case_index"].shape if hasattr(batch["case_index"], "shape") else type(batch["case_index"]))
print("axis_id      :", batch["axis_id"].shape if hasattr(batch["axis_id"], "shape") else type(batch["axis_id"]))
print("slice_index  :", batch["slice_index"].shape if hasattr(batch["slice_index"], "shape") else type(batch["slice_index"]))

# NEW DEBUG
print("window_ids   :", batch["window_ids"].shape)  # [B, Wn]

In [ ]:
# ============================================================
# BASIC ACTIVATION HELPER
# ============================================================

def get_activation(name: str = "relu"):
    name = name.lower()
    if name == "relu":
        return nn.ReLU(inplace=True)
    elif name == "leakyrelu":
        return nn.LeakyReLU(0.2, inplace=True)
    elif name == "gelu":
        return nn.GELU()
    elif name == "silu":
        return nn.SiLU(inplace=True)
    else:
        return nn.ReLU(inplace=True)


# ============================================================
# PAPER-STYLE 2D U-NET BLOCK
# ============================================================

class PaperUNetBlock2D(nn.Module):
    """
    Paper-style U-Net block with fixed internal channel width f.
    Input : [B, f, H, W]
    Output: [B, f, H, W]
    """
    def __init__(self, f: int, dropout: float = 0.0, activation: str = "leakyrelu"):
        super().__init__()

        def conv3(in_ch, out_ch, stride):
            layers = [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                get_activation(activation),
            ]
            if dropout > 0.0:
                layers.append(nn.Dropout2d(dropout))
            return nn.Sequential(*layers)

        def deconv4(in_ch, out_ch):
            layers = [
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=True),
                get_activation(activation),
            ]
            if dropout > 0.0:
                layers.append(nn.Dropout2d(dropout))
            return nn.Sequential(*layers)

        self.c1 = conv3(f, f, stride=2)
        self.c2 = conv3(f, f, stride=2)
        self.c3 = conv3(f, f, stride=1)
        self.c4 = conv3(f, f, stride=2)
        self.c5 = conv3(f, f, stride=1)

        self.u1 = deconv4(f,   f)
        self.u2 = deconv4(2*f, f)
        self.u3 = deconv4(2*f, f)

        self.out = nn.Conv2d(2*f, f, kernel_size=3, stride=1, padding=1, bias=True)

    def forward(self, x):
        x_in = x

        s1 = self.c1(x_in)
        s2 = self.c2(s1)
        s3 = self.c3(s2)

        b = self.c4(s3)
        b = self.c5(b)

        x = self.u1(b)
        x = torch.cat([x, s3], dim=1)

        x = self.u2(x)
        x = torch.cat([x, s1], dim=1)

        x = self.u3(x)
        x = torch.cat([x, x_in], dim=1)

        x = self.out(x)
        return x


# ============================================================
# TIME FOURIER FEATURES
# ============================================================

def time_fourier_features(t: torch.Tensor, n_freq: int = 8):
    """
    t: [B, T] in normalized range [0, 1]
    returns: [B, T, 1 + 2*n_freq]
    """
    feats = [t.unsqueeze(-1)]
    for k in range(n_freq):
        w = (2.0 ** k) * torch.pi
        feats.append(torch.sin(w * t).unsqueeze(-1))
        feats.append(torch.cos(w * t).unsqueeze(-1))
    return torch.cat(feats, dim=-1)


# ============================================================
# UNIFIED 2D SLICE SEQUENCE MODEL
# ============================================================

class UnifiedSliceUNet(nn.Module):

    def __init__(
        self,
        in_channels: int = 17,
        window_size: int = 3,
        base_width: int = 64,
        time_dim: int = 64,
        time_n_freq: int = 8,
        dropout: float = 0.0,
        activation: str = "leakyrelu"
    ):
        super().__init__()

        self.in_channels = in_channels
        self.window_size = window_size
        self.base_width = base_width
        self.time_dim = time_dim
        self.time_n_freq = time_n_freq

        fourier_in_dim = 1 + 2 * time_n_freq

        # SAME encoder as before
        self.lift = nn.Conv2d(in_channels, base_width, kernel_size=1, bias=True)

        self.unet1 = PaperUNetBlock2D(base_width, dropout, activation)
        self.unet2 = PaperUNetBlock2D(base_width, dropout, activation)
        self.unet3 = PaperUNetBlock2D(base_width, dropout, activation)

        self.act_mid = get_activation(activation)

        # NEW: slice fusion layer
        self.slice_fuse = nn.Sequential(
            nn.Conv2d(base_width * window_size, base_width, kernel_size=1, bias=False),
            nn.BatchNorm2d(base_width),
            get_activation(activation),
        )

        # time branch (UNCHANGED)
        self.time_mlp = nn.Sequential(
            nn.Linear(fourier_in_dim, time_dim),
            get_activation(activation),
            nn.Linear(time_dim, time_dim),
            get_activation(activation),
        )

        self.fuse = nn.Sequential(
            nn.Conv2d(base_width + time_dim, base_width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_width),
            get_activation(activation),

            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_width),
            get_activation(activation),

            nn.Conv2d(base_width, 1, kernel_size=1, bias=True)
        )

    # --------------------------------------------------------
    # Encode ONE slice (unchanged logic)
    # --------------------------------------------------------
    def encode_one_slice(self, X_slice):
        x = self.lift(X_slice)
        x = self.unet1(x)
        x = self.act_mid(x)
        x = self.unet2(x)
        x = self.act_mid(x)
        x = self.unet3(x)
        return x   # [B, f -> base_width, H, W]

    # --------------------------------------------------------
    # NEW: Encode WINDOW of slices
    # --------------------------------------------------------
    def encode_window(self, X_window):
        # X_window: [B, Wn, Cin, H, W]

        B, Wn, Cin, H, W = X_window.shape

        feats = []
        for w in range(Wn):
            f_w = self.encode_one_slice(X_window[:, w])   # [B,f,H,W]
            feats.append(f_w)

        feat_cat = torch.cat(feats, dim=1)   # [B, Wn*f, H, W]
        feat = self.slice_fuse(feat_cat)     # [B, f, H, W]

        return feat

    # --------------------------------------------------------
    # FORWARD (UPDATED)
    # --------------------------------------------------------
    def forward(self, X_window, times):

        if times.dim() == 1:
            times = times.unsqueeze(0).expand(X_window.shape[0], -1)

        B, Wn, Cin, H, W = X_window.shape
        Tn = times.shape[1]

        # NEW: multi-slice encoding
        feat = self.encode_window(X_window)   # [B, f, H, W]

        # expand spatial encoding across time
        feat_bt = feat.unsqueeze(1).expand(B, Tn, self.base_width, H, W)

        # time embedding
        t_feat = time_fourier_features(times, n_freq=self.time_n_freq)
        t_emb = self.time_mlp(t_feat)
        t_emb = t_emb.unsqueeze(-1).unsqueeze(-1).expand(B, Tn, self.time_dim, H, W)

        # fuse spatial + time
        z = torch.cat([feat_bt, t_emb], dim=2)
        z = z.reshape(B * Tn, self.base_width + self.time_dim, H, W)

        y = self.fuse(z)
        y = y.reshape(B, Tn, H, W)

        return y

In [ ]:
# ============================================================
# MODEL + OPTIMIZER + LOSS / METRICS
# ============================================================

model = UnifiedSliceUNet(
    in_channels=IN_CHANNELS,      # still 17 (per slice)
    window_size=WINDOW_SIZE,      # NEW
    base_width=UNET_FEATURES,
    time_dim=TIME_DIM,
    time_n_freq=TIME_N_FREQ,
    dropout=UNET_DROPOUT,
    activation=ACTIVATION_FUNCTION
).to(DEVICE)

opt = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------
def mse_loss(pred, target):
    return torch.mean((pred - target) ** 2)


def temporal_smoothness_loss(pred):
    if pred.shape[1] < 2:
        return pred.new_tensor(0.0)
    return torch.mean((pred[:, 1:] - pred[:, :-1]) ** 2)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2) + 1e-12)


def denorm_field(x, y_mean, y_std):
    return x * y_std + y_mean


def rmse_physical(pred, target, y_mean, y_std):
    pred_phys = denorm_field(pred, y_mean, y_std)
    tgt_phys  = denorm_field(target, y_mean, y_std)
    return torch.sqrt(torch.mean((pred_phys - tgt_phys) ** 2) + 1e-12)


print(model)


# ============================================================
# UPDATED RUN EPOCH
# ============================================================

def run_epoch(model, loader, train: bool, smoothness_weight: float = 0.0):

    model.train(train)

    total_loss = 0.0
    total_rmse = 0.0
    total_rmse_phys = 0.0
    n = 0

    for batch in loader:

        # ----------------------------
        # NEW INPUT
        # ----------------------------
        X_window = batch["X_window"].to(DEVICE, non_blocking=True)   # [B,Wn,Cin,H,W]
        times    = batch["times"].to(DEVICE, non_blocking=True)      # [B,T]
        Y        = batch["Y"].to(DEVICE, non_blocking=True)          # [B,T,H,W]

        if train:
            opt.zero_grad(set_to_none=True)

        # ----------------------------
        # MODEL FORWARD
        # ----------------------------
        pred = model(X_window, times)                                # [B,T,H,W]

        # ----------------------------
        # LOSS
        # ----------------------------
        loss_main = mse_loss(pred, Y)
        loss_smooth = temporal_smoothness_loss(pred)
        loss = loss_main + smoothness_weight * loss_smooth

        if train:
            loss.backward()
            opt.step()

        # ----------------------------
        # METRICS
        # ----------------------------
        with torch.no_grad():
            r = rmse(pred, Y)
            r_phys = rmse_physical(
                pred, Y,
                y_mean=stats["y_mean"],
                y_std=stats["y_std"]
            )

        bs = X_window.shape[0]   # changed
        total_loss += float(loss.detach()) * bs
        total_rmse += float(r.detach()) * bs
        total_rmse_phys += float(r_phys.detach()) * bs
        n += bs

    return {
        "loss": total_loss / max(n, 1),
        "rmse": total_rmse / max(n, 1),
        "rmse_physical": total_rmse_phys / max(n, 1),
    }

In [ ]:
# ============================================================
# OPTIONAL TORCH / OPTIMIZER SANITY CHECK
# ============================================================
print("Torch version:", torch.__version__)
importlib.import_module("torch._utils")

_ = optim.Adam(
    [torch.nn.Parameter(torch.randn(2, requires_grad=True))],
    lr=1e-3
)
print("Adam OK")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")
print("\n")


In [ ]:
# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

ckpt_dir = RUN_DIR / f"checkpoints_unified_slice_unet_{TARGET}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

best_val = float("inf")
history = {
    "epoch": [],
    "train_loss": [],
    "train_rmse": [],
    "train_rmse_physical": [],
    "val_loss": [],
    "val_rmse": [],
    "val_rmse_physical": [],
}

# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_metrics = run_epoch(
        model,
        dl_train,
        train=True
    )
    val_metrics = run_epoch(
        model,
        dl_val,
        train=False
    )

    dt = time.time() - t0

    tr_loss = train_metrics["loss"]
    tr_rmse = train_metrics["rmse"]
    tr_rmse_phys = train_metrics["rmse_physical"]

    va_loss = val_metrics["loss"]
    va_rmse = val_metrics["rmse"]
    va_rmse_phys = val_metrics["rmse_physical"]

    history["epoch"].append(epoch)
    history["train_loss"].append(tr_loss)
    history["train_rmse"].append(tr_rmse)
    history["train_rmse_physical"].append(tr_rmse_phys)
    history["val_loss"].append(va_loss)
    history["val_rmse"].append(va_rmse)
    history["val_rmse_physical"].append(va_rmse_phys)

    print(
        f"epoch {epoch:03d} | "
        f"train loss {tr_loss:.4e} rmse {tr_rmse:.4e} rmse_phys {tr_rmse_phys:.4e} | "
        f"val loss {va_loss:.4e} rmse {va_rmse:.4e} rmse_phys {va_rmse_phys:.4e} | "
        f"{dt:.1f}s"
    )

    # --------------------------------------------------------
    # Save best checkpoint using validation loss
    # --------------------------------------------------------
    if va_loss < best_val:
        best_val = va_loss
        ckpt_path = ckpt_dir / "best.pt"

        torch.save(
            {
                "model": model.state_dict(),
                "opt": opt.state_dict(),
                "epoch": epoch,
                "best_val": best_val,

                # dataset / normalization info
                "stats": stats,
                "meta": meta,
                "times": times_np,
                "target": TARGET,

                # model config
                "in_channels": IN_CHANNELS,
                "window_size": WINDOW_SIZE,
                "base_width": UNET_FEATURES,
                "time_dim": TIME_DIM,
                "time_n_freq": TIME_N_FREQ,
                "dropout": UNET_DROPOUT,
                "activation": ACTIVATION_FUNCTION,

                # training / loss config
                "smoothness_weight": SMOOTHNESS_WEIGHT,
                "grad_weight": GRAD_WEIGHT,
                "interface_weight": INTERFACE_WEIGHT,
                "interface_center": INTERFACE_CENTER,
                "interface_sigma": INTERFACE_SIGMA,
                "interface_alpha": INTERFACE_ALPHA,

                # optional
                "history": history,
            },
            ckpt_path
        )
        print("  saved:", ckpt_path)

# ============================================================
# SAVE FINAL CHECKPOINT + TRAINING HISTORY
# ============================================================

final_ckpt_path = ckpt_dir / "last.pt"
torch.save(
    {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "epoch": EPOCHS,
        "best_val": best_val,

        "stats": stats,
        "meta": meta,
        "times": times_np,
        "target": TARGET,

        "in_channels": IN_CHANNELS,
        "window_size": WINDOW_SIZE,
        "base_width": UNET_FEATURES,
        "time_dim": TIME_DIM,
        "time_n_freq": TIME_N_FREQ,
        "dropout": UNET_DROPOUT,
        "activation": ACTIVATION_FUNCTION,

        "smoothness_weight": SMOOTHNESS_WEIGHT,
        "grad_weight": GRAD_WEIGHT,
        "interface_weight": INTERFACE_WEIGHT,
        "interface_center": INTERFACE_CENTER,
        "interface_sigma": INTERFACE_SIGMA,
        "interface_alpha": INTERFACE_ALPHA,

        "history": history,
    },
    final_ckpt_path
)

print("Final checkpoint saved:", final_ckpt_path)

In [ ]:
def get_axis_pad(meta, axis):
    pad_h = int(meta.get(f"pad_h_{axis}", 0))
    pad_w = int(meta.get(f"pad_w_{axis}", 0))
    return pad_h, pad_w


def unpad_2d(arr, pad_h=0, pad_w=0):
    
    H, W = arr.shape
    h_end = H - pad_h if pad_h > 0 else H
    w_end = W - pad_w if pad_w > 0 else W
    return arr[:h_end, :w_end]

In [ ]:
@torch.no_grad()
def plot_slice_timeseries(
    model,
    dataset,
    stats,
    case_id,
    axis="x",
    slice_idx=0,
    ncols=5,
    heat_ch=0,
    meta=None,
    crop_padding=True,
):
    """
    Per-slice inference plot with row 0 = all 6 BC faces.
    Uses the actual BC channels:
      4  bc_xmin
      5  bc_xmax
      6  bc_ymin
      7  bc_ymax
      8  bc_zmin
      9  bc_zmax
    """

    model.eval()

    # --------------------------------------------------------
    # find sample
    # --------------------------------------------------------
    match_idx = None
    for i, s in enumerate(dataset.samples):
        if isinstance(s, dict):
            s_case  = s.get("case_id", s.get("case_index"))
            s_axis  = s.get("axis_name")
            s_slice = s.get("slice_idx", s.get("slice_index"))
        else:
            s_case, s_axis, s_slice = s

        if int(s_case) == int(case_id) and s_axis == axis and int(s_slice) == int(slice_idx):
            match_idx = i
            break

    if match_idx is None:
        raise ValueError(f"No sample found for case_id={case_id}, axis='{axis}', slice_idx={slice_idx}")

    sample = dataset[match_idx]

    # --------------------------------------------------------
    # input
    # --------------------------------------------------------
    X_window = sample["X_window"][None].to(DEVICE)   # [1,Wn,Cin,H,W]
    times_in = sample["times"][None].to(DEVICE)      # [1,T]

    Y_true = sample["Y_raw"].cpu().numpy()           # [T,H,W]

    pred = model(X_window, times_in)[0].cpu().numpy()
    pred = pred * stats["y_std"] + stats["y_mean"]

    err = np.abs(pred - Y_true)

    # --------------------------------------------------------
    # center input slice
    # --------------------------------------------------------
    X_window_np = sample["X_window"].cpu().numpy()   # [Wn,Cin,H,W]
    center_id = X_window_np.shape[0] // 2
    X_center = X_window_np[center_id]                # [Cin,H,W]

    # --------------------------------------------------------
    # time selection
    # --------------------------------------------------------
    Tn = Y_true.shape[0]
    idxs = np.linspace(0, Tn - 1, min(ncols, Tn)).round().astype(int)

    if "times_raw" in sample:
        times_phys = sample["times_raw"].cpu().numpy()
    else:
        times_phys = sample["times"].cpu().numpy()

    # --------------------------------------------------------
    # heat map
    # --------------------------------------------------------
    if not (0 <= heat_ch < X_center.shape[0]):
        raise ValueError(f"heat_ch={heat_ch} out of range")

    Qmap = X_center[heat_ch]

    # --------------------------------------------------------
    # BC faces from actual channels
    # --------------------------------------------------------
    if X_center.shape[0] < 10:
        raise ValueError(f"Expected at least 10 channels, got {X_center.shape[0]}")

    #bc_faces = {
    #    "xmin": X_center[4],
    #    "xmax": X_center[5],
    #    "ymin": X_center[6],
    #    "ymax": X_center[7],
    #    "zmin": X_center[8],
    #    "zmax": X_center[9],
    #}
    bc_faces = {
        "Left": X_center[4],
        "Right": X_center[5],
        "Front": X_center[6],
        "Back": X_center[7],
        "Bottom": X_center[8],
        "Top": X_center[9],
    }
    bc_order = ["Left", "Right", "Front", "Back", "Bottom", "Top"]

    # --------------------------------------------------------
    # crop padding
    # --------------------------------------------------------
    pad_h, pad_w = 0, 0
    if crop_padding and meta is not None:
        pad_h, pad_w = get_axis_pad(meta, axis)

        Qmap   = unpad_2d(Qmap, pad_h, pad_w)
        Y_true = np.stack([unpad_2d(y, pad_h, pad_w) for y in Y_true], axis=0)
        pred   = np.stack([unpad_2d(y, pad_h, pad_w) for y in pred], axis=0)
        err    = np.stack([unpad_2d(y, pad_h, pad_w) for y in err], axis=0)

        for k in bc_faces:
            bc_faces[k] = unpad_2d(bc_faces[k], pad_h, pad_w)

    # --------------------------------------------------------
    # plotting
    # --------------------------------------------------------
    fig, axs = plt.subplots(5, len(idxs), figsize=(3 * len(idxs), 12))

    if len(idxs) == 1:
        axs = np.array(axs).reshape(5, 1)

    # row 0: BC faces
    for j in range(len(idxs)):
        axs[0, j].axis("off")

    for j, face_name in enumerate(bc_order[:len(idxs)]):
        axs[0, j].imshow(bc_faces[face_name], origin="lower", cmap="viridis")
        axs[0, j].set_title(face_name)
        axs[0, j].axis("off")

    # remaining rows
    for j, ti in enumerate(idxs):
        axs[1, j].imshow(Qmap, origin="lower", cmap="Reds")
        axs[1, j].set_title("Q")
        axs[1, j].axis("off")

        axs[2, j].imshow(Y_true[ti], origin="lower", cmap="inferno")
        axs[2, j].set_title(f"True-{TARGET} @ {float(times_phys[ti]):.0f}s")
        axs[2, j].axis("off")

        axs[3, j].imshow(pred[ti], origin="lower", cmap="inferno")
        axs[3, j].set_title(f"Pred-{TARGET}")
        axs[3, j].axis("off")

        axs[4, j].imshow(err[ti], origin="lower", cmap="magma")
        axs[4, j].set_title("Error")
        axs[4, j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
@torch.no_grad()
def infer_and_plot_from_ckpt(
    ckpt_path,
    npz_path,
    split="test",
    case_index=0,
    axis="x",
    slice_idx=0,
    ncols=5,
    heat_ch=0,
):
    # ---------------- LOAD CKPT ----------------
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    stats = ckpt["stats"]
    times = np.asarray(ckpt["times"], dtype=np.float32)
    target = ckpt["target"]

    window_size = int(ckpt.get("window_size", 3))

    # ---------------- LOAD NPZ ----------------
    npz = np.load(npz_path, allow_pickle=True)

    Xstatic_x = npz["Xstatic_x"].astype(np.float32)   # [C,S,Cin,H,W]
    Y_x       = npz["Y_x"].astype(np.float32)         # [C,S,T,2,H,W]

    Xstatic_y = npz["Xstatic_y"].astype(np.float32)
    Y_y       = npz["Y_y"].astype(np.float32)

    Xstatic_z = npz["Xstatic_z"].astype(np.float32)
    Y_z       = npz["Y_z"].astype(np.float32)

    meta = npz["meta"].item() if "meta" in npz.files else {}

    # ---------------- SPLITS ----------------
    splits_path = Path(npz_path).parent / "splits.json"
    with open(splits_path, "r", encoding="utf-8") as f:
        splits = json.load(f)

    split_ids = splits[split]

    if not (0 <= case_index < len(split_ids)):
        raise IndexError(
            f"case_index={case_index} out of range for split='{split}' with {len(split_ids)} cases"
        )

    # ---------------- DATASET ----------------
    ds = Unified3DSliceDataset(
        Xstatic_x, Y_x,
        Xstatic_y, Y_y,
        Xstatic_z, Y_z,
        times=times,
        ids=np.array(split_ids, dtype=np.int64),
        stats=stats,
        target=target,
        window_size=window_size,   # NEW
    )
    ds.times_raw = times

    # ---------------- MODEL ----------------
    model = UnifiedSliceUNet(
        in_channels=ckpt["in_channels"],
        window_size=window_size,                       # NEW
        base_width=ckpt["base_width"],
        time_dim=ckpt.get("time_dim", 64),
        time_n_freq=ckpt.get("time_n_freq", 8),       # add this too
        dropout=ckpt.get("dropout", 0.0),
        activation=ckpt.get("activation", "leakyrelu"),
    ).to(DEVICE)

    model.load_state_dict(ckpt["model"])
    model.eval()

    # ---------------- PLOT ----------------
    plot_slice_timeseries(
        model=model,
        dataset=ds,
        stats=stats,
        case_id=int(split_ids[case_index]),
        axis=axis,
        slice_idx=slice_idx,
        ncols=ncols,
        heat_ch=heat_ch,
        meta=meta,
        crop_padding=True,
    )

    return model

In [ ]:
# ------------------------------------------------------------
# Load dataset once (for slice count info)
# ------------------------------------------------------------
npz = np.load(r"runs/dataset_run_20260408-145632/slice_dataset_3d.npz", allow_pickle=True)

Xstatic_x = npz["Xstatic_x"]
Xstatic_y = npz["Xstatic_y"]
Xstatic_z = npz["Xstatic_z"]

slice_counts = {
    "x": Xstatic_x.shape[1],
    "y": Xstatic_y.shape[1],
    "z": Xstatic_z.shape[1],
}

# ------------------------------------------------------------
# LOOP (SAFE VERSION)
# ------------------------------------------------------------
for axis in ["x", "y", "z"]:

    max_slices = slice_counts[axis]
    n_plot = min(6, max_slices)   # reduce plots (recommended)

    print(f"\n===== AXIS: {axis.upper()} =====")

    for i in range(n_plot):

        print(f"{axis}_slice-{i}")

        infer_and_plot_from_ckpt(
            ckpt_path=r"runs/dataset_run_20260408-145632/checkpoints_unified_slice_unet_T/best.pt",
            npz_path=r"runs/dataset_run_20260408-145632/slice_dataset_3d.npz",
            split="test",
            case_index=0,
            axis=axis,
            slice_idx=i,
            ncols=10,
            heat_ch=0,
        )

In [ ]:
print(model)